# Advanced Problems: Named Tuples as Alternatives to Dictionaries

This notebook contains advanced practice problems with full solutions.

Topic focus:

- Creating `namedtuple` classes dynamically from dictionaries
- Preserving field order
- Safely unpacking dictionaries using keyword arguments
- Handling missing keys with defaults
- Validating field names
- Converting heterogeneous dictionaries into immutable records
- Using `getattr`, `_asdict`, `_replace`, and `_fields`


In [1]:
from collections import namedtuple
from pprint import pprint

## Problem 1: Safe Conversion of a Dictionary to a Named Tuple

Write a function called `dict_to_namedtuple` that takes:

- a dictionary `d`
- a class name `typename`
- an optional flag `sort_fields`

The function should return a named tuple instance containing the dictionary data.

Requirements:

1. The dictionary must have string keys only.
2. The named tuple must be constructed using `**d`, not `*d.values()`.
3. If `sort_fields=True`, fields should be sorted alphabetically.
4. If `sort_fields=False`, field order should follow the dictionary's insertion order.
5. Raise a `TypeError` if any key is not a string.

Example:

```python
person = {'name': 'Ada', 'age': 36, 'language': 'Python'}
record = dict_to_namedtuple(person, 'Person')
record.name
```


### Solution 1

In [2]:
def dict_to_namedtuple(d, typename='Record', *, sort_fields=False):
    """
    Convert a dictionary with string keys into a named tuple instance.

    Parameters
    ----------
    d : dict
        Source dictionary.
    typename : str
        Name of the generated named tuple class.
    sort_fields : bool
        If True, sort fields alphabetically. Otherwise, preserve insertion order.

    Returns
    -------
    namedtuple instance
    """
    if not all(isinstance(key, str) for key in d.keys()):
        raise TypeError('All dictionary keys must be strings.')

    fields = sorted(d.keys()) if sort_fields else d.keys()
    Record = namedtuple(typename, fields)

    return Record(**d)

In [3]:
person = {'name': 'Ada', 'age': 36, 'language': 'Python'}

record = dict_to_namedtuple(person, 'Person')
record

Person(name='Ada', age=36, language='Python')

In [4]:
record.name, record.age, record.language

('Ada', 36, 'Python')

In [5]:
sorted_record = dict_to_namedtuple(person, 'PersonSorted', sort_fields=True)
sorted_record._fields

('age', 'language', 'name')

### Why this works

The important detail is that we use:

```python
Record(**d)
```

instead of:

```python
Record(*d.values())
```

Keyword unpacking matches values to fields by name, so the data does not get corrupted if key order changes.

## Problem 2: Detect the Bug Caused by Positional Unpacking

You are given two dictionaries with the same keys but different insertion order.

Create a named tuple class from the first dictionary's keys. Then show why using `*dict.values()` is dangerous.

Finally, fix the problem using keyword unpacking.


### Solution 2

In [6]:
employee_1 = {
    'id': 101,
    'name': 'Grace Hopper',
    'department': 'Engineering'
}

employee_2 = {
    'department': 'Engineering',
    'id': 102,
    'name': 'Alan Turing'
}

Employee = namedtuple('Employee', employee_1.keys())
Employee._fields

('id', 'name', 'department')

In [7]:
# Dangerous: values are assigned positionally, not by key name.
bad_employee = Employee(*employee_2.values())
bad_employee

Employee(id='Engineering', name=102, department='Alan Turing')

In [8]:
# Correct: values are assigned by keyword.
good_employee = Employee(**employee_2)
good_employee

Employee(id=102, name='Alan Turing', department='Engineering')

### Key lesson

`*dict.values()` depends on order.

`**dict` depends on field names.

When converting dictionaries into named tuples, `**dict` is the safer and more explicit approach.

## Problem 3: Convert Heterogeneous Dictionaries into Uniform Records

You are given a list of dictionaries where not every dictionary has the same keys.

Write a function called `records_from_dicts` that:

1. Accepts an iterable of dictionaries.
2. Finds the union of all keys.
3. Creates a named tuple class using the sorted union of keys.
4. Assigns a default value for missing fields.
5. Returns a list of named tuple instances.

Use `None` as the default value unless another default is supplied.


### Solution 3

In [9]:
def records_from_dicts(dicts, typename='Record', *, default=None):
    """
    Convert an iterable of dictionaries into a list of uniform named tuple records.

    Missing keys are filled using the supplied default value.
    """
    dicts = list(dicts)

    if not dicts:
        return []

    if not all(isinstance(d, dict) for d in dicts):
        raise TypeError('All items must be dictionaries.')

    keys = set().union(*(d.keys() for d in dicts))

    if not all(isinstance(key, str) for key in keys):
        raise TypeError('All dictionary keys must be strings.')

    fields = sorted(keys)
    Record = namedtuple(typename, fields)
    Record.__new__.__defaults__ = (default,) * len(Record._fields)

    return [Record(**d) for d in dicts]

In [10]:
raw_sales = [
    {'region': 'EU', 'sales': 1200},
    {'region': 'US', 'sales': 1800, 'manager': 'Dana'},
    {'region': 'APAC', 'manager': 'Lee'},
    {'sales': 900}
]

sales_records = records_from_dicts(raw_sales, 'SalesRecord')
sales_records

[SalesRecord(manager=None, region='EU', sales=1200),
 SalesRecord(manager='Dana', region='US', sales=1800),
 SalesRecord(manager='Lee', region='APAC', sales=None),
 SalesRecord(manager=None, region=None, sales=900)]

In [11]:
sales_records[0]._fields

('manager', 'region', 'sales')

### Important design choice

The function sorts keys before creating the named tuple class. This avoids unpredictable field order caused by using a set directly.

## Problem 4: Dynamic Attribute Access with Defaults

Dictionaries provide `.get(key, default)`.

Named tuples do not have `.get()` by default.

Write a function called `nt_get` that mimics dictionary `.get()` behavior for named tuple instances.

Requirements:

1. Accept a named tuple instance.
2. Accept a field name.
3. Accept an optional default value.
4. Return the field value if it exists.
5. Return the default value if it does not exist.


### Solution 4

In [12]:
def nt_get(record, field_name, default=None):
    """
    Dictionary-style get() for named tuple instances.
    """
    return getattr(record, field_name, default)

In [13]:
sale = sales_records[1]
sale

SalesRecord(manager='Dana', region='US', sales=1800)

In [14]:
nt_get(sale, 'manager'), nt_get(sale, 'currency', 'USD')

('Dana', 'USD')

### Why this matters

This is useful when the field name is stored in a variable.

For example:

```python
field = 'manager'
getattr(record, field)
```

This works, while:

```python
record.field
```

looks for a literal field named `field`.

## Problem 5: Validate and Rename Invalid Dictionary Keys

`namedtuple` field names must be valid Python identifiers.

The following dictionary cannot be converted directly:

```python
{'first name': 'Guido', 'class': 'Python', '2fa_enabled': True}
```

Write a function called `safe_namedtuple_from_dict` that:

1. Accepts a dictionary.
2. Creates a named tuple using `rename=True`.
3. Returns both the named tuple instance and a mapping from original keys to final field names.

This is useful when incoming data contains invalid or reserved field names.


### Solution 5

In [15]:
def safe_namedtuple_from_dict(d, typename='SafeRecord'):
    """
    Convert a dictionary into a named tuple even when keys are invalid identifiers.

    Returns
    -------
    tuple
        (record, field_mapping)

    field_mapping maps original dictionary keys to actual named tuple field names.
    """
    if not all(isinstance(key, str) for key in d.keys()):
        raise TypeError('All dictionary keys must be strings.')

    original_keys = list(d.keys())
    values = list(d.values())

    Record = namedtuple(typename, original_keys, rename=True)
    record = Record(*values)

    field_mapping = dict(zip(original_keys, Record._fields))

    return record, field_mapping

In [16]:
bad_keys = {
    'first name': 'Guido',
    'class': 'Python',
    '2fa_enabled': True
}

record, mapping = safe_namedtuple_from_dict(bad_keys, 'User')

record, mapping

(User(_0='Guido', _1='Python', _2=True),
 {'first name': '_0', 'class': '_1', '2fa_enabled': '_2'})

### Note

When `rename=True` is used, invalid field names are automatically replaced with positional names such as `_0`, `_1`, and `_2`.

This keeps construction safe, but you may lose meaningful attribute names unless you also keep a mapping.

## Problem 6: Convert Records Back to Dictionaries and Apply Immutable Updates

Named tuples are immutable.

Given a named tuple instance:

1. Convert it back to a dictionary.
2. Create an updated copy using `_replace`.
3. Show that the original record is unchanged.


### Solution 6

In [17]:
Employee = namedtuple('Employee', 'id name department salary')

employee = Employee(
    id=1,
    name='Margaret Hamilton',
    department='Software Engineering',
    salary=150000
)

employee

Employee(id=1, name='Margaret Hamilton', department='Software Engineering', salary=150000)

In [18]:
employee_dict = employee._asdict()
employee_dict

{'id': 1,
 'name': 'Margaret Hamilton',
 'department': 'Software Engineering',
 'salary': 150000}

In [19]:
promoted_employee = employee._replace(salary=175000)

employee, promoted_employee

(Employee(id=1, name='Margaret Hamilton', department='Software Engineering', salary=150000),
 Employee(id=1, name='Margaret Hamilton', department='Software Engineering', salary=175000))

### Key lesson

`_replace` does not mutate the original named tuple.

It creates a new named tuple instance with selected fields changed.

## Problem 7: Build a Small Data Normalization Pipeline

You receive raw API-style data as a list of dictionaries.

Some records are missing fields. Some contain extra fields. You want to normalize the records into named tuples.

Write a function called `normalize_api_records` that:

1. Accepts a list of dictionaries.
2. Builds a named tuple class from all observed keys.
3. Sorts fields alphabetically.
4. Uses `'UNKNOWN'` as the default missing value.
5. Returns the named tuple class and the normalized records.
6. Allows dynamic lookup of any field using a helper function.


### Solution 7

In [20]:
def normalize_api_records(records, typename='APIRecord', *, default='UNKNOWN'):
    """
    Normalize heterogeneous API records into a uniform named tuple structure.

    Returns the generated named tuple class and a list of instances.
    """
    records = list(records)

    if not records:
        Record = namedtuple(typename, [])
        return Record, []

    if not all(isinstance(record, dict) for record in records):
        raise TypeError('Every record must be a dictionary.')

    keys = set().union(*(record.keys() for record in records))

    if not all(isinstance(key, str) for key in keys):
        raise TypeError('Every key must be a string.')

    fields = sorted(keys)

    Record = namedtuple(typename, fields)
    Record.__new__.__defaults__ = (default,) * len(Record._fields)

    normalized = [Record(**record) for record in records]

    return Record, normalized


def lookup(record, field, default=None):
    """
    Dynamic field lookup for normalized named tuple records.
    """
    return getattr(record, field, default)

In [21]:
api_data = [
    {'id': 1, 'name': 'Alice', 'role': 'admin'},
    {'id': 2, 'name': 'Bob'},
    {'id': 3, 'role': 'editor', 'active': True},
    {'name': 'Charlie', 'active': False}
]

APIRecord, normalized = normalize_api_records(api_data)

APIRecord._fields, normalized

(('active', 'id', 'name', 'role'),
 [APIRecord(active='UNKNOWN', id=1, name='Alice', role='admin'),
  APIRecord(active='UNKNOWN', id=2, name='Bob', role='UNKNOWN'),
  APIRecord(active=True, id=3, name='UNKNOWN', role='editor'),
  APIRecord(active=False, id='UNKNOWN', name='Charlie', role='UNKNOWN')])

In [22]:
for record in normalized:
    print(record)

APIRecord(active='UNKNOWN', id=1, name='Alice', role='admin')
APIRecord(active='UNKNOWN', id=2, name='Bob', role='UNKNOWN')
APIRecord(active=True, id=3, name='UNKNOWN', role='editor')
APIRecord(active=False, id='UNKNOWN', name='Charlie', role='UNKNOWN')


In [23]:
field_name = 'role'

[lookup(record, field_name) for record in normalized]

['admin', 'UNKNOWN', 'editor', 'UNKNOWN']

## Problem 8: Advanced Challenge — Strict Schema vs Flexible Schema

Write a function called `schema_records` that supports two modes:

### Flexible mode

- Uses the union of all keys.
- Missing values are filled with a default.

### Strict mode

- Requires every dictionary to have exactly the same keys.
- Raises a `ValueError` if any dictionary has missing or extra keys.

The function should return a list of named tuple records.


### Solution 8

In [24]:
def schema_records(dicts, typename='Record', *, strict=False, default=None):
    """
    Convert dictionaries into named tuple records using strict or flexible schema rules.

    Parameters
    ----------
    dicts : iterable of dict
    typename : str
    strict : bool
        If True, all dictionaries must have exactly the same keys.
    default : object
        Default value for missing fields in flexible mode.
    """
    dicts = list(dicts)

    if not dicts:
        return []

    if not all(isinstance(d, dict) for d in dicts):
        raise TypeError('All items must be dictionaries.')

    all_keys = set().union(*(d.keys() for d in dicts))

    if not all(isinstance(key, str) for key in all_keys):
        raise TypeError('All keys must be strings.')

    if strict:
        expected_keys = set(dicts[0].keys())

        for index, d in enumerate(dicts, start=1):
            current_keys = set(d.keys())

            if current_keys != expected_keys:
                missing = expected_keys - current_keys
                extra = current_keys - expected_keys

                raise ValueError(
                    f'Record {index} does not match schema. '
                    f'Missing: {missing}. Extra: {extra}.'
                )

        fields = sorted(expected_keys)
    else:
        fields = sorted(all_keys)

    Record = namedtuple(typename, fields)

    if not strict:
        Record.__new__.__defaults__ = (default,) * len(Record._fields)

    return [Record(**d) for d in dicts]

In [25]:
flexible_data = [
    {'a': 1, 'b': 2},
    {'a': 3},
    {'a': 4, 'b': 5, 'c': 6}
]

schema_records(flexible_data, 'FlexibleRecord', strict=False, default='N/A')

[FlexibleRecord(a=1, b=2, c='N/A'),
 FlexibleRecord(a=3, b='N/A', c='N/A'),
 FlexibleRecord(a=4, b=5, c=6)]

In [26]:
strict_data = [
    {'a': 1, 'b': 2},
    {'a': 3, 'b': 4},
    {'a': 5, 'b': 6}
]

schema_records(strict_data, 'StrictRecord', strict=True)

[StrictRecord(a=1, b=2), StrictRecord(a=3, b=4), StrictRecord(a=5, b=6)]

In [27]:
# Uncomment this cell to see the strict validation error.
# schema_records(flexible_data, 'BrokenStrictRecord', strict=True)

## Summary of Best Practices

1. Prefer `Record(**d)` over `Record(*d.values())`.
2. Validate that dictionary keys are strings before creating named tuple fields.
3. Sort fields when creating records from a union of keys.
4. Use defaults when dictionaries are heterogeneous.
5. Use `getattr(record, field, default)` for dynamic field access.
6. Use `_asdict()` when converting a named tuple back to a dictionary.
7. Use `_replace()` when creating an updated copy.
8. Avoid creating a new named tuple class for every single dictionary if many dictionaries share a schema.
9. Consider `rename=True` when field names may be invalid Python identifiers.
10. Choose strict schema validation when inconsistent data should be treated as an error.
